# 5 · Run All Scenarios (sweep) on FABRIC

Execute **every** scenario in `validation/scenarios/` end-to-end on the slice — single
scenarios plus the expanded `sweep_distance` (6 pts) and `sweep_attenuation` (9 pts) — and
for each point run the full 4-way cross-validation (QFabric measured + QFabric-sim +
SeQUeNCe + NetSquid). Then plot QBER and secure key rate vs distance/attenuation.

> ⏳ **This is the long one.** ~18 points × (switch reconfigure + BB84 + 3 simulator runs)
> ≈ 30–60+ min of slice time. Results are written to `results/all_scenarios.json` after
> **every** point, so an interruption keeps what's done.

**Prerequisite:** `01_setup_slice` (slice up, data-plane IPs) completed. The simulator
envs are built in Step 1 below (one-time).

### At a glance
- **Purpose:** batch-run **every** scenario in `validation/scenarios/` (singles + expanded sweeps) and produce the QBER/key-rate-vs-distance/attenuation figures.
- **Inputs:** `SLICE_NAME`, NetSquid creds (Step 1).
- **Outputs:** `results/all_scenarios.json` (rewritten after every point) + sweep figures + an all-points table.
- **Runs on / runtime:** FABRIC JupyterHub; **~30–60+ min** (18 points × switch reconfigure + BB84 + 3 simulator runs). Set `cross_validate=False` for a faster BB84-only sweep.
- **If something fails:** the sweep is resumable (partial `all_scenarios.json` is kept) and one bad point won't abort the rest; high-loss points may legitimately show the measured backend as SKIPPED while the simulators still populate the row.

## Configuration (match notebook 1)

In [1]:
SLICE_NAME = 'qfabric-bb84-2'

In [2]:
import os, sys, json
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR)); sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
import deploy_fabric as df

fablib = df.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


ID,d1035024-ba8f-4203-b038-632073474a82
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-07-09 04:09:19 +0000
Lease Start (UTC),2026-07-07 16:08:19 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


ID,d1035024-ba8f-4203-b038-632073474a82
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-07-09 04:09:19 +0000
Lease Start (UTC),2026-07-07 16:08:19 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


## Step 1 — Build simulator envs on the nodes (one-time)
Set your netsquid.org credentials first so NetSquid installs on the bob node.

In [3]:
# os.environ['NETSQUID_USER'] = '...'
# os.environ['NETSQUID_PASS'] = '...'
df.setup_sim_envs(slice_obj)


=== Setting up simulator envs on FABRIC nodes (one-time, several min) ===
  [switch] QFabric-sim (.venv-qsim)...
Reading package lists...
Building dependency tree...
Reading state information...
python3-pip is already the newest version (22.0.2+dfsg-1ubuntu0.7).
python3-venv is already the newest version (3.10.6-1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 27 not upgraded.
  [alice] SeQUeNCe 1.0 on Python 3.12 (.venv-seq)...
Reading package lists...
Building dependency tree...
Reading state information...
software-properties-common is already the newest version (0.99.22.9).
0 upgraded, 0 newly installed, 0 to remove and 27 not upgraded.
Hit:1 http://nova.clouds.archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://nova.clouds.archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jamm

## Step 2 — Run every scenario (this takes a while)
Reconfigures the switch loss threshold per point, runs BB84, then the 4-way comparison.
Set `cross_validate=False` for a faster BB84-only sweep.

In [4]:
rows = df.run_all_scenarios_on_fabric(slice_obj, cross_validate=True)
print(f'\nCompleted {len(rows)} scenario points.')

Discovered scenarios in validation/scenarios -> 38 runs

##### [1/38] baseline_1km : baseline_1km (dist=1.0 km, atten=0.2 dB/km, F=0.98) #####

=== Configuring switch (threshold=193305371) ===
  Switch interfaces: enp7s0 (Alice), enp8s0 (Bob)
  Alice MAC: 2E:C6:67:6F:9F:11
  Bob MAC:   06:C1:A6:14:84:EA
  Switch Alice-side MAC: 32:CA:F2:F8:00:90
  Switch Bob-side MAC:   32:F4:25:17:27:B3
  Compiling P4 program...
  Starting BMv2...
  Configuring tables...
  Switch configured and running

=== Running BB84 protocol ===
  Bob data-plane IP: 10.10.1.2 (for classical channel)
  Alice data iface:  enp7s0
  Bob data iface:    enp7s0
  Cleaning up previous runs...
  Ensuring deps on alice...
  Ensuring deps on bob...
  Starting Bob...
  Alice MAC args:  --dst-mac '32:CA:F2:F8:00:90' --src-mac '2E:C6:67:6F:9F:11'
  Starting Alice...
  Waiting for BB84 to complete...
  Alice finished
  Alice output: Alice: Sending 100000 photons on enp7s0
Alice: Finished sending 100000 photons
Alice: Connecting 

## Step 3 — Sweep figures: QBER & key rate vs distance / attenuation
One line per backend; loads `results/all_scenarios.json` (so this cell also works on a
previously-saved sweep without re-running Step 2).

In [5]:
import matplotlib.pyplot as plt
%matplotlib inline
rows = json.loads((PROJECT_DIR / 'results' / 'all_scenarios.json').read_text())

def _ok(b):
    return (not b.get('extra', {}).get('error')) and b.get('sifted_bits', 0) > 0

def plot_group(group, xkey, xlabel):
    pts = sorted([r for r in rows if r['group'] == group], key=lambda r: r[xkey])
    if not pts:
        print(f'(no points for {group})'); return
    platforms = []
    for r in pts:
        for b in r['backends']:
            if b['platform'] not in platforms: platforms.append(b['platform'])
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for p in platforms:
        xs, q, s = [], [], []
        for r in pts:
            b = next((b for b in r['backends'] if b['platform'] == p and _ok(b)), None)
            if b:
                xs.append(r[xkey]); q.append(b['qber'] * 100); s.append(b.get('secure_key_rate', 0))
        if xs:
            axes[0].plot(xs, q, marker='o', label=p)
            axes[1].plot(xs, s, marker='o', label=p)
    axes[0].set(xlabel=xlabel, ylabel='QBER (%)', title=f'QBER vs {xlabel}')
    axes[1].set(xlabel=xlabel, ylabel='Secure key rate (bits/photon)', title=f'Key rate vs {xlabel}')
    for ax in axes: ax.grid(alpha=0.3); ax.legend()
    fig.suptitle(group); plt.tight_layout(); plt.show()

plot_group('sweep_dark_count', 'dark_count_rate_hz', 'dark count rate (cps)')
plot_group('sweep_efficiency', 'detector_efficiency', 'detector efficiency')
plot_group('sweep_polarization', 'polarization_fidelity', 'polarization fidelity')

KeyError: 'dark_count_rate_hz'

## Step 4 — All-scenarios summary table

Every point (singles + sweep points) with each backend's QBER. A blank/`NaN`
means that backend was SKIPPED for that point (e.g. a measured run that produced
no key at high loss, or an uninstalled simulator).

In [ ]:
import pandas as pd
tbl = []
for r in rows:
    row = {'group': r['group'], 'scenario': r['scenario'],
           'distance_km': r['distance_km'], 'atten': r['attenuation_db_per_km']}
    for b in r['backends']:
        row[b['platform']] = round(b['qber'], 4) if _ok(b) else None
    tbl.append(row)
df = pd.DataFrame(tbl)
print(f'{len(df)} scenario points')
df

---
All scenarios executed. `results/all_scenarios.json` holds every point × backend for
further analysis or for the paper figures.